# Variable-Coefficient Poisson Equation Demo
Solve the PDE:
\begin{align}
-\nabla \cdot k \nabla u &= f \quad \text{in} \; \Omega \\
-\mathbf{n} \cdot k \nabla u &= g \quad \text{on} \; \Gamma_n \\
u &= u_d  \quad \text{on} \; \Gamma_d,
\end{align}
where $k$ is the conductivity tensor (a SPD matrix in $\mathbb{R}^{d \times d}$)

In [ ]:
import variable_coefficient_poisson
import mesh
import sparse_matrices
import numpy as np

In [ ]:
import tri_mesh_viewer

In [ ]:
m = mesh.Mesh('../3rdparty/MeshFEM/misc/examples/meshes/square_hole.off', degree=1)

In [ ]:
# Construct conductivity tensors as the identity tensor field
N = m.embeddingDimension
ks = np.ones((m.numElements(), (N * (N + 1)) // 2))
ks[:, N:] = 0

In [ ]:
# Nodal values of the forcing function
f = np.zeros(m.numNodes()) 
f[0] = 0

In [ ]:
v = tri_mesh_viewer.Viewer(m, wireframe=True)
v.show()

In [ ]:
neumannElements = []
neumannFluxes = []
dirichletNodes = [0, 1]
dirichletValues = [0, 1]
vp = variable_coefficient_poisson.construct(m, ks, f, neumannElements, neumannFluxes, dirichletNodes, dirichletValues)

In [ ]:
from matplotlib import pyplot as plt
A_sp = vp.A.toSymmetryMode(vp.A.symmetry_mode.NONE).toSciPy()
plt.spy(A_sp, markersize=2)

In [ ]:
# Solve the linear system
solver = sparse_matrices.CholeskyFactorizer()
solver.factorize(vp.A)
u = solver.solve(vp.b)

In [ ]:
v.update(scalarField=u)